# Max-t permutation test: baseline vs. post-patching

Which channels, in which frequency bands, are reconstructed differently after patching? One paired test per channel × band, corrected for multiple comparisons over all of them at once.

**Data.** The post-stimulus patch of every trial (the *combined* contrast of `plasticity_plotting.ipynb`), metrics of `time_window`, all folds of `model_name` (requires `run_inference.py --postpatch` outputs).

**Method: subject-level paired sign-flip permutation test with max-t correction**
1. Per subject, session, channel and band (+ all bands): average the log of the trials' error (`plot_data_processing.calc_subject_channel_band_means`). Errors are positive and right-skewed, and session differences in them are closer to proportional than additive. Trials in which a channel was interpolated are left out of that channel's mean.
2. Per subject: post-patching − baseline in every cell, i.e. a log ratio. This gives a subjects × cells matrix and one paired t = mean / (SD / √n) per cell.
3. Null distribution: flip the sign of each subject's whole row at random, recompute all t-values and keep the maximum |t|, `n_permutations` times (`core.permutation_test.max_t_sign_flip_test`). The observed signs count as one permutation, so p = (1 + #{max |t| ≥ |t_obs|}) / (n_permutations + 1).

Flipping whole rows keeps the correlation between cells (neighbouring channels, overlapping bands, metrics), so the correction is exact for this dependence, and much less conservative than Bonferroni.

**Design decisions beyond the specification**
- **Step-down max-t (Westfall & Young) decides significance.** The single-step test compares every |t| with the maximum over all cells. The step-down test compares the *k*-th largest |t| only with the maximum over the cells with smaller or equal |t|. It controls the family-wise error rate just as strongly and is never less powerful. Both p-values are reported.
- **All metrics form one family.** MAE and level error are tested in a single max-t over every metric × band × channel cell, so the family-wise error rate holds over every claim of this notebook. The two metrics are strongly correlated, which the max-t takes into account, so the second metric costs little power. Set `metrics` to MAE alone to test only MAE.
- **Log per metric.** MAE: mean of the trial logs (the log of the geometric mean), robust to single outlier trials. Level error: log of the subject mean. A trial's level error is the absolute value of a difference close to 0, so single trials come close to 0 and their logs have a long negative tail. A subject's mean over many trials stays well away from 0.
- **"all bands" stays in the family.** It largely repeats the four bands, and redundant (correlated) cells cost the max-t little.
- **Cell-level, not cluster-level inference.** A cluster permutation test is more sensitive to spatially extended effects, but it only supports the claim that a cluster contains *some* effect, not that a particular channel or band changed. The question here is about particular channels and bands.
- **Missing values.** A subject without a value in a cell (the channel interpolated in every trial of one session) is left out of that cell only. Sign-flipping whole rows stays a valid permutation. Cells with fewer than `min_subjects` subjects are left out of the family.

**Assumption.** Under H0 every subject's differences are symmetric around 0 (exchangeable signs). The log makes this much more plausible: see the skewness check.

**Reading the results.** Positive t = the error *increases* after patching (worse reconstruction). Changes are reported in % of the baseline geometric mean, 100 · (exp(mean log difference) − 1).

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import skew

from core import io, plotting, settings, utils
from core import plot_data_processing as data_processor
from core.permutation_test import max_t_sign_flip_test

In [ ]:
model_name = "masking_10_19"
conditions = ["global"]  # stimulus types, None loads all
time_window = "analysis"  # key of settings.METRIC_WINDOWS whose metrics are tested
exclude_left_patched = True  # excludes the left-eye-dominant subjects of eye_dominance.csv
exclude_right_patched_lefties = True  # excludes the left-eye-dominant subjects of rightpatch_eye_dominance.csv (right-eye patching from STMD35 on)
exclude_button_press = False  # excludes the subjects of the button-press paradigm (before NO_PRESS_FROM_SUBJECT)

# tested metrics, all in one family, and how each one's trials are averaged into a subject mean:
# "mean_of_logs" (log of the geometric mean) or "log_of_mean" (log of the arithmetic mean)
metrics = {"mae": "mean_of_logs", "level_error": "log_of_mean"}
n_permutations = 10000
alpha = 0.05  # family-wise error rate
seed = 0  # seed of the random sign flips, for reproducible p-values
min_subjects = 10  # cells with fewer subjects with values in both sessions are left out of the family

In [ ]:
bands = list(settings.FREQ_BANDS.keys())
band_keys = [*bands, data_processor.ALL_BANDS]
CH_NAMES = [name.replace("EEG ", "").replace("-REF", "") for name in settings.CHORDER_STANDARD]

exclude_subjects = list(settings.EXCLUDED_SUBJECTS)
if exclude_left_patched:
    exclude_subjects.extend(utils.get_subjects_by_eye_dominance("left", settings.DATA_ROOT / "eye_dominance.csv"))
if exclude_right_patched_lefties:
    exclude_subjects.extend(utils.get_subjects_by_eye_dominance("left", settings.DATA_ROOT / "rightpatch_eye_dominance.csv"))
if exclude_button_press:
    exclude_subjects.extend(subj for subj in io.get_stored_subject_names(settings.DATA_ROOT)
                            if utils.subject_paradigm(subj, settings.NO_PRESS_FROM_SUBJECT) == "press")

# file names carry the window and the excluded groups, the folder the model and the stimulus types (as in plasticity_plotting.ipynb)
excluded_groups = [group for group, excluded in [("left", exclude_left_patched), ("rightlefties", exclude_right_patched_lefties),
                                                 ("buttonpress", exclude_button_press)] if excluded]
file_suffix = f"window_{time_window}" + (f"_excl_{'_'.join(excluded_groups)}" if excluded_groups else "")
plot_root = settings.PLOT_INFERENCE / model_name / ("_".join(conditions) if conditions else "all_conditions") / "max_t_permutation"

### Load the post-stimulus patch of all folds

In [ ]:
# every numbered fold folder of the model holds the baseline and post-patching reconstructions of its test trials
fold_dirs = sorted((d for d in (settings.INFERENCE_DIR / model_name).iterdir() if d.name.isdigit()), key=lambda d: int(d.name))
trials = {"baseline": [], "postpatch": []}
for fold_dir in fold_dirs:
    for session, session_trials in trials.items():
        session_trials.extend(io.load_inference_trials(fold_dir / f"{session}_reconstruction.h5", conditions=conditions,
                                                       exclude_subjects=exclude_subjects, window=time_window, stim_patch="post"))
print(f"{len(fold_dirs)} folds: {len(trials['baseline'])} baseline and {len(trials['postpatch'])} post-patching trials")

### Subject × cell matrix of log differences

In [ ]:
def paired_rows(session_means, subjects):
    # rows of the given subjects, in that order, of the means and trial counts of calc_subject_channel_band_means
    session_subjects, means, counts = session_means
    rows = [session_subjects.index(subject) for subject in subjects]
    return means[rows], counts[rows]


# per metric and session: subjects, log subject means (subjects, channels, bands) and the number of trials in every mean
log_means = {metric: {session: data_processor.calc_subject_channel_band_means(session_trials, metric, bands, log=log)
                      for session, session_trials in trials.items()}
             for metric, log in metrics.items()}

# paired subjects: subjects with trials in both sessions (the same for every metric)
first_metric = log_means[next(iter(metrics))]
b_subjects, p_subjects = first_metric["baseline"][0], first_metric["postpatch"][0]
subjects = sorted(set(b_subjects) & set(p_subjects))

# post-patching - baseline per subject and cell, every metric's (subjects, channels, bands) flattened into columns
diffs_by_metric, trial_counts = {}, []
for metric, sessions in log_means.items():
    b_means, b_counts = paired_rows(sessions["baseline"], subjects)
    p_means, p_counts = paired_rows(sessions["postpatch"], subjects)
    diffs_by_metric[metric] = p_means - b_means
    trial_counts.extend([b_counts, p_counts])
trial_counts = np.stack(trial_counts)
diffs =np.concatenate([d.reshape(len(subjects), -1) for d in diffs_by_metric.values()], axis=1)
cells = pd.MultiIndex.from_product([list(metrics), CH_NAMES, band_keys], names=["metric", "channel", "band"])

n_per_cell = np.isfinite(diffs).sum(axis=0)
print(f"{len(subjects)} subjects with both sessions, {diffs.shape[1]} cells ({len(metrics)} metrics x {len(CH_NAMES)} channels x {len(band_keys)} bands)")
print(f"left out, only one session: {sorted(set(b_subjects) ^ set(p_subjects)) or 'none'}")
print(f"subjects per cell: {n_per_cell.min()} - {n_per_cell.max()}, fewest trials in a subject mean: {trial_counts[trial_counts > 0].min()}, "
      f"subject means without any trial (channel interpolated in the whole session): {np.sum(trial_counts == 0)}")

### Check: symmetry of the subject differences

The sign-flip test needs the subjects' differences to be symmetric around 0 under H0. Skewness per cell of the differences of the raw subject means (arithmetic means) and of the log differences that are tested. Even for symmetric data, the skewness of one cell scatters with SE ≈ √(6/N), so look at the mean over cells (≈ 0 if symmetric) and compare the median |skewness| with its value for symmetric data.

In [ ]:
print(f"{'metric':12s} {'differences':12s} {'mean skewness':>14s} {'median |skewness|':>18s}")
for metric in metrics:
    raw_means = {session: data_processor.calc_subject_channel_band_means(session_trials, metric, bands) for session, session_trials in trials.items()}
    raw_diffs = paired_rows(raw_means["postpatch"], subjects)[0] - paired_rows(raw_means["baseline"], subjects)[0]
    for label, metric_diffs in [("raw", raw_diffs), ("log", diffs_by_metric[metric])]:
        cell_skewness = np.asarray(skew(metric_diffs.reshape(len(subjects), -1), axis=0, nan_policy="omit"), dtype=float)
        print(f"{metric:12s} {label:12s} {np.nanmean(cell_skewness):+14.2f} {np.nanmedian(np.abs(cell_skewness)):18.2f}")
print(f"median |skewness| of symmetric data with N={len(subjects)}: about {0.674 * np.sqrt(6 / len(subjects)):.2f}")

### Permutation test

In [ ]:
result = max_t_sign_flip_test(diffs, n_permutations=n_permutations, seed=seed, min_subjects=min_subjects)
results = pd.DataFrame({
    "n": result.n,
    "mean_log_diff": result.mean,
    "change_%": 100 * np.expm1(result.mean),  # change of the geometric mean (mean_of_logs) or of the mean (log_of_mean)
    "d_z": result.t / np.sqrt(result.n),
    "t": result.t,
    "p_single_step": result.p_single_step,
    "p_step_down": result.p_step_down,
}, index=cells)
results["significant"] = results["p_step_down"] <= alpha

critical_t = result.critical_t(alpha)
print(f"{results['t'].notna().sum()} of {len(results)} cells tested, {n_permutations} sign flips, smallest possible p = {1 / (n_permutations + 1):.1e}")
print(f"critical |t| of the single-step test at FWER {alpha}: {critical_t:.2f}")
print(f"significant cells: {results['significant'].sum()} (step-down), {(results['p_single_step'] <= alpha).sum()} (single step)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(result.null_max, bins=60, color="lightgray", edgecolor="gray", label=f"max |t| over all cells, {n_permutations} sign flips")
ax.axvline(critical_t, color="black", linestyle="--", label=f"critical |t| = {critical_t:.2f} (FWER {alpha}, single step)")
observed = np.abs(results["t"].dropna().to_numpy())
ax.plot(observed, np.zeros_like(observed), "|", color="tab:red", markersize=20, label="observed |t| of every cell")
ax.set_xlabel("|t|")
ax.set_ylabel("permutations")
ax.set_title(f"Null distribution of the maximum |t| (N={len(subjects)} subjects), stimulus type: {', '.join(conditions) if conditions else 'all'}")
ax.legend()
plotting.save_button(fig, save_dir=plot_root, default_name=f"null_distribution_{file_suffix}")
plt.show()

### Significant cells

Sorted by |t|, which is also the order of the step-down p-values. `change_%`: change of the subjects' error after patching, `d_z`: mean / SD of the subject log differences.

In [ ]:
significant = results[results["significant"]].sort_values("t", key=np.abs, ascending=False)
display(significant.drop(columns="significant").round({"mean_log_diff": 4, "change_%": 2, "d_z": 2, "t": 2, "p_single_step": 4, "p_step_down": 4}))

In [ ]:
# all cells, tested or not, as a CSV next to the figures
io.save_report(results.reset_index().to_dict("records"), f"max_t_results_{file_suffix}.csv", plot_root)

### Overview: t of every cell

Rows: bands, columns: channels, one panel per metric. `*`: significant (step-down, FWER `alpha` over all panels). Blank cells were left out of the family.

In [ ]:
t_limit = np.nanmax(np.abs(results["t"]))
fig, axes = plt.subplots(len(metrics), 1, figsize=(14, 1 + 2.8 * len(metrics)), squeeze=False, gridspec_kw={"hspace": 0.6})
for ax, metric in zip(axes[:, 0], metrics):
    metric_results = results.loc[metric]
    t_grid = metric_results["t"].unstack("channel").loc[band_keys, CH_NAMES].to_numpy(dtype=float)
    significant_grid = metric_results["significant"].unstack("channel").loc[band_keys, CH_NAMES].to_numpy(dtype=bool)
    im = ax.imshow(t_grid, cmap="RdBu_r", vmin=-t_limit, vmax=t_limit, aspect="auto")
    for row, col in zip(*np.nonzero(significant_grid)):
        ax.text(col, row, "*", ha="center", va="center", fontsize=14)
    ax.set_xticks(range(len(CH_NAMES)), CH_NAMES, rotation=90)
    ax.set_yticks(range(len(band_keys)), band_keys)
    ax.set_title(f"{metric} ({metrics[metric].replace('_', ' ')})")
fig.colorbar(im, ax=axes, shrink=0.8, label="t (post-patching - baseline)")
fig.suptitle(f"Paired t per channel and band (N={len(subjects)} subjects), stimulus type: {', '.join(conditions) if conditions else 'all'}")
plotting.save_button(fig, save_dir=plot_root, default_name=f"t_overview_{file_suffix}")
plt.show()

### Topomaps

Per metric: change in % (top) and t (bottom) of every band, one symmetric colour scale per row. White dots: significant channels (step-down, FWER `alpha` over all metrics, bands and channels). Channels left out of the family are not drawn.

In [ ]:
info = data_processor.create_topomap_info()  # 10-20 labels at the positions of the recording HydroCel electrodes, in CH_NAMES order
mask_params = dict(marker="o", markerfacecolor="white", markeredgecolor="black", linewidth=0, markersize=7)

for metric in metrics:
    metric_results = results.loc[metric]
    tested = metric_results[metric_results["t"].notna()]
    rows = [("change_%", "change (%)", np.nanmax(np.abs(tested["change_%"]))), ("t", "t", t_limit)]

    fig, axes = plt.subplots(len(rows), len(band_keys), figsize=(4 * len(band_keys), 7.5))
    for col, band in enumerate(band_keys):
        band_results = metric_results.xs(band, level="band").loc[CH_NAMES]  # channel order of info
        keep = np.flatnonzero(band_results["t"].notna())
        images = []
        for row, (column, _, limit) in enumerate(rows):
            im, _ = mne.viz.plot_topomap(band_results[column].to_numpy(dtype=float)[keep], mne.pick_info(info, keep), axes=axes[row, col],
                                         show=False, vlim=(-limit, limit), cmap="RdBu_r",
                                         mask=band_results["significant"].to_numpy(dtype=bool)[keep], mask_params=mask_params)
            images.append(im)
        axes[0, col].set_title(band)
    for row, (_, label, _) in enumerate(rows):
        fig.colorbar(images[row], ax=axes[row], shrink=0.8, label=label)  # every panel of a row shares its colour scale
    fig.suptitle(f"{metric}: post-patching - baseline (N={len(subjects)} subjects), stimulus type: {', '.join(conditions) if conditions else 'all'}", y=1.0)
    plotting.save_button(fig, save_dir=plot_root, default_name=f"{metric}_topomaps_{file_suffix}")
    plt.show()

### Check: comparison with MNE

`mne.stats.permutation_t_test` implements the single-step test on complete data. On the cells without missing subjects, both give the same t-values and, with different random sign flips, p-values that differ only by Monte Carlo error (SE ≈ √(p(1−p)/n_permutations), about 0.002 at p = 0.05).

In [ ]:
complete = np.isfinite(diffs).all(axis=0) & results["t"].notna().to_numpy()
t_mne, p_mne, _ = mne.stats.permutation_t_test(diffs[:, complete], n_permutations=n_permutations, seed=seed, verbose=False)
check = max_t_sign_flip_test(diffs[:, complete], n_permutations=n_permutations, seed=seed + 1, min_subjects=min_subjects)
print(f"{complete.sum()} complete cells: t equal: {np.allclose(check.t, t_mne)}, "
      f"largest difference of the single-step p-values: {np.max(np.abs(check.p_single_step - p_mne)):.4f}")